In [3]:

# nos librairies
import pandas as pd
import matplotlib as plt

In [23]:
# affichons notre dataframe clean en fixant le type de la variable jour en datetimes
df = pd.read_csv("../data/covid_hospit_clean.csv", parse_dates=["jour"])
print(df.head())

  dep  sexe       jour  hosp  rea  HospConv  SSR_USLD  autres  rad  dc
0  01     0 2020-03-18     2    0       NaN       NaN     NaN    1   0
1  01     1 2020-03-18     1    0       NaN       NaN     NaN    1   0
2  01     2 2020-03-18     1    0       NaN       NaN     NaN    0   0
3  02     0 2020-03-18    41   10       NaN       NaN     NaN   18  11
4  02     1 2020-03-18    19    4       NaN       NaN     NaN   11   6


## KPI GLOGAUX

In [46]:
df.dtypes

dep                    str
sexe                 int64
jour        datetime64[us]
hosp                 int64
rea                  int64
HospConv           float64
SSR_USLD           float64
autres             float64
rad                  int64
dc                   int64
dtype: object

In [47]:
df.columns

Index(['dep', 'sexe', 'jour', 'hosp', 'rea', 'HospConv', 'SSR_USLD', 'autres',
       'rad', 'dc'],
      dtype='str')

In [48]:
# nous allons afficher un descriptif de nos données

df[["hosp",	"rea","HospConv","SSR_USLD","autres","rad","dc"]].describe().round(2)

,hosp,rea,HospConv,SSR_USLD,autres,rad,dc
count,338245.00,338245.00,228140.00,228140.00,228140.00,338245.00,338245.00
mean,113.80,13.36,61.50,35.98,2.99,2817.62,534.74
std,166.75,28.40,83.83,52.00,5.83,4173.61,730.47
min,0.00,0.00,0.00,0.00,0.00,0.00,0.00
25%,23.00,1.00,15.00,6.00,0.00,500.00,105.00
50%,58.00,4.00,34.00,18.00,1.00,1404.00,278.00
75%,134.00,13.00,73.00,44.00,3.00,3338.00,653.00
max,3281.00,855.00,1115.00,565.00,170.00,48210.00,6463.00


Nous observons qu'il y'a un très grand écart entre le min et le max des différentes variables.
Ce ne sont pas de valeurs abbérantes.
Ici les grands chiffres sont normaux car :

la pandémie a duré 3 ans ;
il y a 338 245 observations ;
les indicateurs sont cumulés quotidiennement.

Si nous prenons par exemple le descriptif de la variable (hosp)
le nombre d'hospitalisé par jour varie de 0 à 3281 personnes avec une moyenne de 113,80 par jour qui repésente le double de sa médiane 58.

La majorité des observations présentent un nombre d'hospitalisations relativement faible, tandis qu'un nombre limité d'observations affiche des niveaux très élevés, ce qui tire la moyenne vers le haut.

De même concernant le nombre de personnes en réanimation (rea), on a un effectif qui varie en 0 et 855 avec une moyenne de 13,36 ce qui traduit que la plupart du temps, les services de réanimation fonctionnaient à un niveau modéré. Certaines périodes ont connu une forte pression hospitalière.

Concernant les décès (dc), on a entre 0 et 6463 décès par jour avec une moyenne de 534.74 et une médiane de 278. ce qui traduit quelques épisodes exceptionnels qui augmentent fortement les valeurs observées.
 


### nous allons chercher les indicateurs pertinents

In [49]:
# nombre d'hospitalisations maximal observé
df[["jour", "hosp"]].max()

jour    2023-03-31 00:00:00
hosp                   3281
dtype: object

In [50]:
# le nombre maximum de patients en réanimation
df[["jour", "rea"]].max()

jour    2023-03-31 00:00:00
rea                     855
dtype: object

In [51]:
# nous pouvons oberver que le jour qui a été le plus critique dans les services hospitaliers en France était:
df.groupby("jour")["hosp"].sum().sort_values(ascending=False). head(10)

jour
2020-11-16    66605
2022-02-07    66417
2022-02-08    66233
2020-11-17    65964
2020-11-15    65759
2022-02-06    65562
2022-02-04    65492
2022-02-09    65297
2022-02-01    65287
2020-11-18    65285
Name: hosp, dtype: int64

In [52]:
# les départements qui ont été les plus touchés par la pandémie cette période sont:
df.groupby("dep")["hosp"].sum().sort_values(ascending=False). head(10)

dep
75    1967155
13    1752745
59    1655530
92    1654392
93    1635225
94    1340362
69    1204623
78    1044302
91     900666
95     833454
Name: hosp, dtype: int64

Les sommes globales des variables hosp, rea, rad et dc produisent des valeurs très élevées. Cette situation est normale car les indicateurs sont observés quotidiennement sur plusieurs années et pour l'ensemble des départements français.

Ces variables représentent des effectifs présents à une date donnée et non des événements uniques. L'agrégation par simple somme conduit donc à compter plusieurs fois les mêmes individus au fil du temps.

Pour obtenir des indicateurs plus pertinents, l'analyse s'orientera vers les tendances temporelles, les valeurs maximales observées et les comparaisons entre départements plutôt que vers les sommes globales.


In [53]:
# nous allons avoir un résultat global du nombre de fois nos différents services ont reçu des personnes pendant cette période
indice_global= df[["hosp", "rea","rad","dc"]].sum()
print(indice_global)

hosp     38493882
rea       4519997
rad     953045503
dc      180873550
dtype: int64
